<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.4_Praktikum_Activation_Function.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.4: Praktikum Fungsi Aktivasi & Arsitektur Multi-Layer Perceptron (MLP)
## Mata Kuliah: AI Academy (INSTIPER)

---

### Tujuan Praktikum:
1. Mengimplementasikan fungsi aktivasi kontinu non-linier (Sigmoid, Tanh, ReLU, Leaky ReLU, Softmax) serta turunan analitiknya menggunakan NumPy.
2. Memvisualisasikan fenomena matematis **Vanishing Gradient** dan **Dying ReLU**.
3. Membangun model forward pass jaringan Multi-Layer Perceptron (MLP) murni secara tensorial.
4. Melatih model MLP untuk klasifikasi multi-kelas 4 fraksi kematangan Tandan Buah Segar (TBS) kelapa sawit.
5. Membuktikan secara empiris bahwa aktivasi linier murni gagal memecahkan masalah non-linier, serta mengeksplorasi ruang fitur tersembunyi (*latent feature space*).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA

# Pengaturan visualisasi publikasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 120

print("[OK] Seluruh pustaka untuk eksplorasi Multi-Layer Perceptron berhasil dimuat!")

## 1. Implementasi Komputasi Fungsi Aktivasi & Turunan Analitik
Di bawah ini kita mendefinisikan fungsi aktivasi dasar beserta turunan pertamanya untuk pembaruan gradien.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def tanh_func(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1.0 - np.tanh(z)**2

def relu(z):
    return np.maximum(0.0, z)

def relu_deriv(z):
    return np.where(z > 0.0, 1.0, 0.0)

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0.0, z, alpha * z)

def leaky_relu_deriv(z, alpha=0.01):
    return np.where(z > 0.0, 1.0, alpha)

def softmax(z):
    # Stabil secara numerik dengan mengurangi nilai maksimum z
    exp_z = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

print("[OK] Seluruh fungsi aktivasi dan turunan analitik berhasil didefinisikan!")

## 2. Visualisasi Kurva Nilai dan Gradien: Memahami Vanishing Gradient
Kita membandingkan respon turunan pertama fungsi aktivasi untuk memahami mengapa Sigmoid memicu *vanishing gradient* pada jaringan dalam.

In [ ]:
z_range = np.linspace(-5.0, 5.0, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Plot Fungsi Aktivasi
ax1.plot(z_range, sigmoid(z_range), label='Sigmoid', color='#e74c3c', lw=2)
ax1.plot(z_range, tanh_func(z_range), label='Tanh', color='#8e44ad', lw=2)
ax1.plot(z_range, relu(z_range), label='ReLU', color='#27ae60', lw=2.2)
ax1.plot(z_range, leaky_relu(z_range, 0.1), label='Leaky ReLU (a=0.1)', color='#2980b9', lw=2, linestyle='--')
ax1.set_title('Nilai Luaran Fungsi Aktivasi $f(z)$', fontweight='bold')
ax1.set_xlabel('Net Input ($z$)')
ax1.set_ylabel('Aktivasi $f(z)$')
ax1.set_ylim(-1.5, 3.5)
ax1.legend()

# Plot Turunan Pertama
ax2.plot(z_range, sigmoid_deriv(z_range), label='Sigmoid Derivative (Maks 0.25)', color='#e74c3c', lw=2)
ax2.plot(z_range, tanh_deriv(z_range), label='Tanh Derivative (Maks 1.00)', color='#8e44ad', lw=2)
ax2.plot(z_range, relu_deriv(z_range), label='ReLU Derivative (Konstan 1.0)', color='#27ae60', lw=2.2)
ax2.plot(z_range, leaky_relu_deriv(z_range, 0.1), label='Leaky ReLU Derivative', color='#2980b9', lw=2, linestyle='--')
ax2.set_title("Profil Gradien Turunan Pertama $f'(z)$", fontweight='bold')
ax2.set_xlabel('Net Input ($z$)')
ax2.set_ylabel('Gradien $df/dz$')
ax2.set_ylim(-0.1, 1.2)
ax2.legend()

plt.tight_layout()
plt.show()

## 3. Dataset Agro-Industri: Sortasi Multi-Kelas Kematangan TBS Sawit
Kita membangkitkan 800 sampel tandan buah segar (TBS) dengan 4 fitur optik:
1. `Indeks_Kemerahan`: Rasio warna merah pada permukaan brondolan.
2. `Rasio_Biru_Hijau`: Penyerapan spektrum biru oleh karotenoid.
3. `Indeks_Kilap_Optik`: Reflektansi spekular minyak permukaan.
4. `Persentase_Brondol_Lepas`: Estimasi brondolan buah yang telah lepas alami dari tandan.

Target: 4 Kelas Kematangan:
- **0**: Mentah (*Unripe*)
- **1**: Kurang Matang (*Underripe*)
- **2**: Matang Sempurna (*Ripe*)
- **3**: Lewat Matang (*Overripe*)

In [ ]:
np.random.seed(42)
n_per_class = 200

# Kelas 0: Mentah (Hitam keunguan, tidak ada brondol lepas)
f0 = np.random.multivariate_normal([0.22, 0.65, 0.15, 0.0], [[0.002, 0, 0, 0], [0, 0.002, 0, 0], [0, 0, 0.001, 0], [0, 0, 0, 0.001]], n_per_class)
y0 = np.zeros(n_per_class, dtype=int)

# Kelas 1: Kurang Matang (Mulai memerah, brondol lepas sedikit)
f1 = np.random.multivariate_normal([0.38, 0.50, 0.35, 0.04], [[0.002, 0, 0, 0], [0, 0.002, 0, 0], [0, 0, 0.001, 0], [0, 0, 0, 0.002]], n_per_class)
y1 = np.ones(n_per_class, dtype=int)

# Kelas 2: Matang Sempurna (Jingga kemerahan mengkilap, 10-25% brondol lepas)
f2 = np.random.multivariate_normal([0.58, 0.32, 0.70, 0.18], [[0.002, 0, 0, 0], [0, 0.002, 0, 0], [0, 0, 0.001, 0], [0, 0, 0, 0.002]], n_per_class)
y2 = np.full(n_per_class, 2, dtype=int)

# Kelas 3: Lewat Matang (Merah gelap/busuk, brondolan rontok masif)
f3 = np.random.multivariate_normal([0.48, 0.25, 0.45, 0.45], [[0.003, 0, 0, 0], [0, 0.002, 0, 0], [0, 0, 0.002, 0], [0, 0, 0, 0.004]], n_per_class)
y3 = np.full(n_per_class, 3, dtype=int)

X_tbs = np.vstack([f0, f1, f2, f3])
y_tbs = np.concatenate([y0, y1, y2, y3])
X_tbs = np.clip(X_tbs, 0.0, 1.0)

feature_cols = ['Indeks_Kemerahan', 'Rasio_Biru_Hijau', 'Indeks_Kilap_Optik', 'Persentase_Brondol_Lepas']
df_tbs = pd.DataFrame(X_tbs, columns=feature_cols)
df_tbs['Kelas_Kematangan'] = y_tbs

print(f"Dataset TBS Sawit berhasil dibuat: {len(df_tbs)} sampel, 4 kelas seimbang.")
print(df_tbs.groupby('Kelas_Kematangan').mean().round(3))

## 4. Pembagian Data dan Standardisasi
Jaringan saraf sangat sensitif terhadap skala fitur, sehingga standardisasi dengan `StandardScaler` mutlak diterapkan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tbs, y_tbs, test_size=0.25, random_state=42, stratify=y_tbs
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Jumlah Sampel Latih : {X_train.shape[0]}")
print(f"Jumlah Sampel Uji   : {X_test.shape[0]}")

## 5. Implementasi Forward Pass Tensorial Mandiri (Pure NumPy)
Di bawah ini kita membuktikan operasi aljabar linier perambatan maju (*Forward Pass*) dari input hingga output Softmax.

In [ ]:
class MiniMLPForward:
    def __init__(self, layer_sizes=[4, 16, 8, 4], seed=42):
        rng = np.random.RandomState(seed)
        self.weights = []
        self.biases = []
        # He Normal initialization
        for i in range(len(layer_sizes) - 1):
            n_in, n_out = layer_sizes[i], layer_sizes[i+1]
            limit = np.sqrt(2.0 / n_in)
            W = rng.normal(0, limit, size=(n_in, n_out))
            b = np.zeros(n_out)
            self.weights.append(W)
            self.biases.append(b)

    def forward(self, X):
        a = X
        # Hidden layers dengan ReLU
        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            z = np.dot(a, W) + b
            a = np.maximum(0.0, z)  # ReLU
        # Output layer dengan Softmax
        W_last, b_last = self.weights[-1], self.biases[-1]
        z_out = np.dot(a, W_last) + b_last
        probs = softmax(z_out)
        return probs

mlp_manual = MiniMLPForward([4, 16, 8, 4])
sample_probs = mlp_manual.forward(X_test_scaled[:3])

print("Contoh Probabilitas Luaran Softmax dari Forward Pass Manual (3 Sampel Pertama):")
for idx, p in enumerate(sample_probs):
    print(f"Sampel #{idx+1}: [Mentah: {p[0]:.3f}, Kurang: {p[1]:.3f}, Matang: {p[2]:.3f}, Lewat: {p[3]:.3f}] -> Jumlah Total: {np.sum(p):.4f}")

## 6. Pembuktian Empiris: Mengapa Aktivasi Linier Gagal Dibanding Non-Linier?
Kita membandingkan 4 model MLP dengan fungsi aktivasi berbeda pada arsitektur tersembunyi `(16, 8)`:
1. `identity` (Fungsi aktivasi linier murni $f(z) = z$)
2. `logistic` (Fungsi Sigmoid)
3. `tanh` (Tangen Hiperbolik)
4. `relu` (Rectified Linear Unit)

In [ ]:
activations = ['identity', 'logistic', 'tanh', 'relu']
results = []

for act in activations:
    mlp = MLPClassifier(
        hidden_layer_sizes=(16, 8),
        activation=act,
        solver='adam',
        max_iter=400,
        random_state=42
    )
    mlp.fit(X_train_scaled, y_train)
    y_pred = mlp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results.append({
        'Fungsi Aktivasi': act.upper(),
        'Akurasi Uji (%)': round(acc * 100, 2),
        'Jumlah Iterasi': mlp.n_iter_,
        'Loss Akhir': round(mlp.loss_, 4)
    })

df_res = pd.DataFrame(results)
print("=== TABEL KOMPARASI PRESTASI FUNGSI AKTIVASI PADA MLP ===")
print(df_res.to_string(index=False))

## 7. Evaluasi Detail Model Terbaik (MLP ReLU)
Kita membedah Confusion Matrix dan Classification Report dari model MLP ReLU untuk menjamin tidak ada buah mentah yang salah dikirim ke sterilizer PKS.

In [ ]:
best_mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=400,
    random_state=42
)
best_mlp.fit(X_train_scaled, y_train)
y_pred_best = best_mlp.predict(X_test_scaled)

target_names = ['Mentah', 'Kurang Matang', 'Matang Sempurna', 'Lewat Matang']

plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix Sortasi Kematangan TBS (MLP ReLU)', fontweight='bold', pad=12)
plt.xlabel('Prediksi Model AI')
plt.ylabel('Status Aktual TBS')
plt.show()

print("Laporan Metrik Klasifikasi Presisi:")
print(classification_report(y_test, y_pred_best, target_names=target_names))

## 8. Visualisasi Ruang Fitur Tersembunyi (*Latent Space Transformation*)
Bagaimana lapisan tersembunyi (*hidden layer*) membengkokkan data sehingga menjadi terpisah secara linier? Kita memproyeksikan data asli vs representasi lapisan tersembunyi ke dalam bidang 2D menggunakan PCA.

In [ ]:
# Reduksi PCA pada fitur asli
pca_raw = PCA(n_components=2)
X_raw_2d = pca_raw.fit_transform(X_test_scaled)

# Reduksi PCA pada aktivasi tersembunyi (Feedforward manual lapisan ke-2)
# Gunakan bobot best_mlp
W1, W2 = best_mlp.coefs_[0], best_mlp.coefs_[1]
b1, b2 = best_mlp.intercepts_[0], best_mlp.intercepts_[1]

h1_act = np.maximum(0, np.dot(X_test_scaled, W1) + b1)
h2_act = np.maximum(0, np.dot(h1_act, W2) + b2)

pca_latent = PCA(n_components=2)
X_latent_2d = pca_latent.fit_transform(h2_act)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Plot Fitur Asli
scatter1 = ax1.scatter(X_raw_2d[:, 0], X_raw_2d[:, 1], c=y_test, cmap='viridis', edgecolors='black', s=60, alpha=0.8)
ax1.set_title('(a) Ruang Fitur Asli (Masih Saling Berhimpit)', fontweight='bold')
ax1.set_xlabel('Principal Component 1')
ax1.set_ylabel('Principal Component 2')

# Plot Ruang Laten Lapisan Tersembunyi
scatter2 = ax2.scatter(X_latent_2d[:, 0], X_latent_2d[:, 1], c=y_test, cmap='viridis', edgecolors='black', s=60, alpha=0.8)
ax2.set_title('(b) Ruang Laten Hidden Layer (Terurai Sempurna)', fontweight='bold')
ax2.set_xlabel('Latent Dimension 1')
ax2.set_ylabel('Latent Dimension 2')

plt.tight_layout()
plt.show()
print("[SIMPULAN]: Lapisan tersembunyi dengan aktivasi ReLU sukses melipat ruang data sehingga 4 klaster terpisah secara linier sempurna!")